# Una red neuronal a mano, con NumPy

**Facsímil 1 · Los cimientos** — capítulos 4 a 7
(la neurona, las capas, la retropropagación y los optimizadores).

Casi todo el mundo ha oído que las redes neuronales son «cajas negras» llenas de magia. Este cuaderno
desmonta esa idea construyendo una red **desde cero**, solo con NumPy: sin PyTorch, sin TensorFlow, sin
ninguna pieza que esconda lo que pasa por dentro. Al terminar habrás escrito tú mismo el mecanismo que
mueve a *todos* los modelos modernos, desde un clasificador de imágenes hasta un LLM, y lo habrás puesto
a prueba en varios problemas y con varias configuraciones. No hay magia: hay álgebra, una función que
curva y una regla de tres para repartir el error.

### Qué vas a aprender
- Qué es **exactamente** una neurona, escrita como una fórmula, y por qué la función de activación es
  imprescindible.
- Por qué una sola neurona **no puede** resolver el XOR, y qué tuvo paralizada a la IA por esa
  limitación.
- Cómo una **capa oculta** rompe la barrera, y qué significa geométricamente (lo verás con varias
  imágenes).
- La **retropropagación**, derivada paso a paso, sin caja negra.
- Cómo influyen, **con experimentos que ejecutas tú**: el número de neuronas, la función de activación
  (sigmoide, tanh, ReLU), el ritmo de aprendizaje y el azar de la inicialización.
- A entrenar la red en un **problema realista** (dos «lunas» entrelazadas) y a ver su frontera curva.

### Cómo se usa este cuaderno
Las celdas vienen **sin ejecutar**: pulsa ▶ (o `Mayús+Enter`) en cada una, de arriba abajo, y verás
nacer cada resultado y cada gráfico. Lee el texto antes de ejecutar; la gracia no es que «salga», sino
entender *por qué* sale.

### Cuánto cuesta
Unos 20 minutos con calma. CPU; sin GPU ni cuentas.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# En Colab no hace falta instalar nada (NumPy, Matplotlib y scikit-learn ya vienen).
# En tu ordenador:  pip install numpy matplotlib scikit-learn
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)
print("NumPy", np.__version__, "· listo para empezar.")


## 1. Qué es una neurona, en una fórmula

Despojada de mística, una **neurona artificial** es una función que recibe varios números y devuelve uno.
Hace tres cosas:

1. **Pondera y suma:** $z = w_1 x_1 + \dots + w_n x_n + b = \mathbf{w}\cdot\mathbf{x} + b$. Los pesos
   $w_i$ son la importancia de cada entrada; $b$ (el *sesgo*) desplaza el resultado.
2. **Activa:** pasa $z$ por una función no lineal $f$. Sin ella, encadenar neuronas no serviría de nada
   (componer cosas lineales da algo lineal). La **sigmoide** aplasta cualquier número a $(0,1)$:
   $\sigma(z) = 1/(1+e^{-z})$.
3. **Devuelve** $y = f(z)$.

Los **pesos** y el **sesgo** son los únicos números que la red aprende. Entrenar es buscar los que hacen
que la salida se parezca a lo que queremos. Dibujemos la sigmoide, porque su forma explica medio cuaderno.


In [ ]:
def sigmoide(z): return 1.0 / (1.0 + np.exp(-z))

zz = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 2.8))
plt.plot(zz, sigmoide(zz), color="black")
plt.axhline(0.5, ls=":", color="#9c9c9c"); plt.axvline(0, ls=":", color="#9c9c9c")
plt.title("La sigmoide: aplasta cualquier numero a (0, 1)")
plt.xlabel("z = w·x + b"); plt.ylabel("σ(z)"); plt.tight_layout(); plt.show()
print("Lejos del centro decide con seguridad (0 o 1); cerca de z=0 'duda' (~0.5).")
print("Esa transicion SUAVE es lo que permite calcular cuanto mejora la salida si movemos un peso.")


## 2. El problema: dos sensores y una decisión que no es una raya

Un riego automático con dos lecturas binarias: **¿tierra seca?** (0/1) y **¿va a llover?** (0/1). La regla
sensata: **riega solo cuando las dos señales discrepan**. Seca y sin lluvia → riega; húmeda y con lluvia
→ no; seca y con lluvia → espera; húmeda y sin lluvia → no. Esa regla de «exactamente una» es el **XOR**.

Tiene una trampa profunda: **no se separa con una sola recta**. Los casos «regar» quedan en esquinas
opuestas de un cuadrado, igual que los «no regar». En 1969, Minsky y Papert lo demostraron y hundieron
durante años el entusiasmo por las redes, porque la pieza de entonces (el perceptrón) solo sabía trazar
rectas. Vamos a revivir ese fracaso... y a superarlo.


In [ ]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)   # 1 = regar
for (s,l), r in zip(X, y):
    print(f"seca={int(s)}  lluvia={int(l)}  ->  {'REGAR' if r[0] else 'no regar'}")

plt.figure(figsize=(3.4,3.4))
for (sx,sy),t in zip(X,y):
    plt.scatter(sx,sy,c="black",marker=("o" if t[0] else "x"),s=200,linewidths=2)
plt.xticks([0,1]); plt.yticks([0,1]); plt.xlim(-.3,1.3); plt.ylim(-.3,1.3)
plt.xlabel("¿seca?"); plt.ylabel("¿lluvia?")
plt.title("'o'=regar, 'x'=no.\n¿una sola recta los separa?"); plt.tight_layout(); plt.show()


## 3. Intento 1: una sola neurona (perceptrón logístico)

Para entrenar necesitamos una **pérdida** (cuánto de mal lo hacemos) y su **gradiente** (hacia dónde
moverla). Usamos la **entropía cruzada binaria**, $L=-[y\log p+(1-y)\log(1-p)]$, que castiga estar seguro
y equivocado. Con sigmoide + entropía cruzada ocurre una simplificación preciosa: la derivada respecto a
$z$ es justo $p-y$ (lo predicho menos lo real). Repetimos el ajuste miles de veces (*descenso de
gradiente*). Mira el techo: no pasará de 2 de 4.


In [ ]:
w = np.random.randn(2,1)*0.5; b = np.zeros((1,1)); lr = 0.5; hist1 = []
for _ in range(5000):
    p = sigmoide(X@w+b); hist1.append(-np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9)))
    dz = (p-y)/len(X); w -= lr*(X.T@dz); b -= lr*dz.sum(0,keepdims=True)
p = sigmoide(X@w+b); ac = int(((p>0.5).astype(float)==y).sum())
print("Probabilidades de la neurona:", np.round(p.ravel(),3))
print(f"Aciertos: {ac} de 4 · perdida final {hist1[-1]:.4f} (no baja de ahi)")


La neurona devuelve probabilidades pegadas a 0.5: se *encoge de hombros*. Veámoslo: dibujamos su
frontera (donde decide 0.5). Será una **recta**, y ninguna recta separa el XOR.


In [ ]:
gx,gy = np.meshgrid(np.linspace(-.3,1.3,200), np.linspace(-.3,1.3,200))
malla = np.c_[gx.ravel(),gy.ravel()]
pm = sigmoide(malla@w+b).reshape(gx.shape)
plt.figure(figsize=(4,3.7))
plt.contourf(gx,gy,pm,levels=[0,.5,1],colors=["#eaeaea","#9c9c9c"])
plt.contour(gx,gy,pm,levels=[.5],colors="black")
for (sx,sy),t in zip(X,y): plt.scatter(sx,sy,c="black",marker=("o" if t[0] else "x"),s=160,linewidths=2)
plt.title("La frontera de UNA neurona es una recta"); plt.xlabel("¿seca?"); plt.ylabel("¿lluvia?")
plt.tight_layout(); plt.show()


Ahí está el muro histórico, dibujado: la recta siempre deja mal clasificado algún punto. La salida no
es «1+1=2»; es un sistema que confiesa, con números, que no puede. La solución llegó al **apilar**
neuronas.


## 4. Por qué una capa oculta lo cambia todo

La idea es **componer**. Con varias neuronas en una capa intermedia (la *oculta*), cada una traza su
recta; una neurona final las combina y, juntas, encierran regiones **curvas**. El *teorema de aproximación
universal* dice que una red con una capa oculta suficientemente grande aproxima casi cualquier función.
Para el XOR bastan **4** neuronas ocultas. La red será $2\to4\to1$, y el paso hacia delante es encadenar
dos veces «pondera, suma, activa»:
$$\mathbf{a}_1=\sigma(\mathbf{x}W_1+\mathbf{b}_1)\ \rightarrow\ p=\sigma(\mathbf{a}_1W_2+b_2)$$


In [ ]:
def init(ni,no): return np.random.randn(ni,no)*np.sqrt(2/ni), np.zeros((1,no))
W1,b1 = init(2,4); W2,b2 = init(4,1)
print("W1:", W1.shape, "(2 entradas -> 4 ocultas) · W2:", W2.shape, "(4 -> 1 salida)")
print("Total de numeros que la red aprende:", W1.size+b1.size+W2.size+b2.size)


## 5. La retropropagación, sin caja negra

Para entrenar necesitamos la culpa de **cada** peso, incluidos los ocultos. La herramienta es la **regla
de la cadena**: encadenar derivadas hacia atrás. En cada época:

1. **Forward:** $z_1=xW_1+b_1$, $a_1=\sigma(z_1)$, $z_2=a_1W_2+b_2$, $p=\sigma(z_2)$.
2. **Error de salida:** $\delta_2=p-y$.
3. **Gradientes última capa:** $\nabla W_2=a_1^\top\delta_2$; se propaga $\delta_1=(\delta_2W_2^\top)\cdot
   \sigma'(z_1)$, con $\sigma'=\sigma(1-\sigma)$.
4. **Gradientes capa oculta:** $\nabla W_1=x^\top\delta_1$.
5. **Actualización:** cada peso baja en la dirección de su gradiente (ritmo `lr`).

Lo escribimos tal cual, guardando además *fotos* de los pesos para luego ver el aprendizaje en cámara lenta.


In [ ]:
W1,b1 = init(2,4); W2,b2 = init(4,1); lr = 1.0; hist = []; fotos = []
for ep in range(8000):
    a1 = sigmoide(X@W1+b1); p = sigmoide(a1@W2+b2)
    hist.append(-np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9)))
    if ep in (0, 200, 800, 7999): fotos.append((ep, W1.copy(),b1.copy(),W2.copy(),b2.copy()))
    d2 = (p-y)/len(X); dW2 = a1.T@d2; db2 = d2.sum(0,keepdims=True)
    d1 = (d2@W2.T)*a1*(1-a1); dW1 = X.T@d1; db1 = d1.sum(0,keepdims=True)
    W2 -= lr*dW2; b2 -= lr*db2; W1 -= lr*dW1; b1 -= lr*db1

def predice(M,W1,b1,W2,b2): return sigmoide(sigmoide(M@W1+b1)@W2+b2)
p = predice(X,W1,b1,W2,b2); ac = int(((p>0.5).astype(float)==y).sum())
print("Probabilidades de la red:", np.round(p.ravel(),3))
print(f"Aciertos: {ac} de 4 · perdida {hist[0]:.3f} -> {hist[-1]:.4f}")


Las cuatro neuronas ocultas se reparten el trabajo, la última las combina, y la frontera deja de ser
una raya. Las probabilidades se van a los extremos: la red **está segura**, y acierta. No programamos
*cómo* resolver el XOR: definimos la arquitectura y la retropropagación encontró los pesos.


## 6. El aprendizaje, en una curva

Cada punto es una época: medir el error y dar un pasito que lo reduce, ocho mil veces. Comparamos la red
(que aprende) con la neurona sola (atascada arriba).


In [ ]:
plt.figure(figsize=(7,3.2))
plt.plot(hist,color="black",label="red 2-4-1 (aprende)")
plt.plot(hist1,color="#9c9c9c",ls="--",label="una neurona (atascada)")
plt.xlabel("epoca"); plt.ylabel("perdida"); plt.title("La red baja; la neurona sola no puede")
plt.legend(); plt.tight_layout(); plt.show()


## 7. El aprendizaje en cámara lenta

Guardamos «fotos» de la red en cuatro momentos. Dibujamos su frontera en cada uno: al principio es casi
una raya inútil; poco a poco se **curva** hasta encerrar bien los cuatro casos. Así se *ve* aprender.


In [ ]:
fig,axes = plt.subplots(1,4,figsize=(14,3.4))
for ax,(ep,W1f,b1f,W2f,b2f) in zip(axes,fotos):
    pm = predice(malla,W1f,b1f,W2f,b2f).reshape(gx.shape)
    ax.contourf(gx,gy,pm,levels=[0,.5,1],colors=["#eaeaea","#9c9c9c"])
    ax.contour(gx,gy,pm,levels=[.5],colors="black",linewidths=1.3)
    for (sx,sy),t in zip(X,y): ax.scatter(sx,sy,c="black",marker=("o" if t[0] else "x"),s=110,linewidths=2)
    ax.set_title(f"epoca {ep}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("De una frontera casi recta (inutil) a una curva que separa el XOR. Eso es entrenar.")


## 8. ¿Qué ve cada neurona oculta?

La red combina cuatro rectas (una por neurona oculta) para encerrar las regiones correctas. Las dibujamos
junto a la frontera final: ahí ves, literalmente, cómo la composición crea la no-linealidad.


In [ ]:
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9,4))
pm = predice(malla,W1,b1,W2,b2).reshape(gx.shape)
ax1.contourf(gx,gy,pm,levels=[0,.5,1],colors=["#eaeaea","#9c9c9c"])
ax1.contour(gx,gy,pm,levels=[.5],colors="black",linewidths=1.5)
for (sx,sy),t in zip(X,y): ax1.scatter(sx,sy,c="black",marker=("o" if t[0] else "x"),s=150,linewidths=2)
ax1.set_title("Frontera curva de la red")
a1m = sigmoide(malla@W1+b1).reshape(200,200,4)
for k in range(4): ax2.contour(gx,gy,a1m[:,:,k],levels=[.5],colors="black",linewidths=1)
for (sx,sy),t in zip(X,y): ax2.scatter(sx,sy,c="black",marker=("o" if t[0] else "x"),s=110,linewidths=2)
ax2.set_title("Las 4 rectas de las neuronas ocultas")
plt.tight_layout(); plt.show()


## 9. Experimento A: ¿cuánta red hace falta?

Entrenamos con 1, 2, 3, 4 y 8 neuronas ocultas, **10 arranques** cada una (el azar de la inicialización
importa), y contamos cuántas veces resuelve el XOR.


In [ ]:
def entrena(no, sem, ep=8000, lr=1.0, act="sig"):
    rng = np.random.default_rng(sem)
    W1 = rng.standard_normal((2,no))*np.sqrt(2/2); b1 = np.zeros((1,no))
    W2 = rng.standard_normal((no,1))*np.sqrt(2/no); b2 = np.zeros((1,1))
    f = {"sig":(sigmoide, lambda a:a*(1-a)),
         "tanh":(np.tanh, lambda a:1-a**2),
         "relu":(lambda z:np.maximum(0,z), lambda a:(a>0).astype(float))}[act]
    fwd, der = f
    for _ in range(ep):
        z1 = X@W1+b1; a1 = fwd(z1); p = sigmoide(a1@W2+b2)
        d2 = (p-y)/len(X); d1 = (d2@W2.T)*der(a1)
        W2 -= lr*(a1.T@d2); b2 -= lr*d2.sum(0,keepdims=True)
        W1 -= lr*(X.T@d1); b1 -= lr*d1.sum(0,keepdims=True)
    p = sigmoide(fwd(X@W1+b1)@W2+b2)
    return int(((p>0.5).astype(float)==y).sum())

print("neuronas | resuelve el XOR (de 10 arranques)")
for n in [1,2,3,4,8]:
    e = sum(entrena(n,s)==4 for s in range(10)); print(f"   {n:>2}    | {'#'*e:<10} {e}/10")


Con **1** neurona nunca (una recta); con **2** a veces falla (capacidad justa + azar); a partir de **4**
casi siempre. Dos lecciones: más capacidad ayuda hasta un punto, y el **azar de la inicialización** es
real (por eso se entrena varias veces o se usan buenas inicializaciones).


## 10. Experimento B: la función de activación

La sigmoide fue la primera, pero hoy casi nadie la usa en capas ocultas: se «satura» (sus gradientes se
hacen diminutos lejos del centro) y aprende despacio. Comparamos **sigmoide**, **tanh** y **ReLU**
($\max(0,z)$, la favorita actual) en el mismo XOR, con 4 neuronas y 10 arranques.


In [ ]:
print("activacion | resuelve el XOR (de 10 arranques, 4 neuronas)")
for act,nombre in [("sig","sigmoide"),("tanh","tanh"),("relu","ReLU")]:
    e = sum(entrena(4,s,act=act)==4 for s in range(10))
    print(f"  {nombre:<8} | {'#'*e:<10} {e}/10")
print("\nSorpresa: en un problema TAN pequeno la sigmoide va de sobra, y ReLU a veces falla por")
print("'neuronas muertas' (una unidad ReLU que cae en zona negativa deja de aprender, y con solo 4")
print("unidades eso pesa). En redes GRANDES y profundas es al reves: ReLU y tanh evitan la saturacion")
print("de la sigmoide (sus gradientes diminutos lejos del centro) y por eso dominan hoy.")


**La lección, matizada.** La elección de activación importa, pero su efecto depende del problema. La
sigmoide aquí brilla porque la red es minúscula; en redes profundas su saturación la hace inviable y se
imponen ReLU y tanh. Moraleja: no hay una activación «mejor» universal, hay una mejor *para tu caso*, y
conviene probar.


## 11. Experimento C: el ritmo de aprendizaje (`lr`)

El `lr` decide el tamaño del paso del descenso de gradiente. Muy pequeño: aprende a paso de tortuga. Muy
grande: rebota y se desestabiliza. Hay una zona buena en medio. Lo medimos con la pérdida final.


In [ ]:
def perdida_final(lr, no=4, ep=4000, sem=0):
    rng = np.random.default_rng(sem)
    W1 = rng.standard_normal((2,no))*np.sqrt(2/2); b1 = np.zeros((1,no))
    W2 = rng.standard_normal((no,1))*np.sqrt(2/no); b2 = np.zeros((1,1)); L = 0
    for _ in range(ep):
        a1 = sigmoide(X@W1+b1); p = sigmoide(a1@W2+b2)
        L = -np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9))
        d2 = (p-y)/len(X); d1 = (d2@W2.T)*a1*(1-a1)
        W2 -= lr*(a1.T@d2); b2 -= lr*d2.sum(0,keepdims=True)
        W1 -= lr*(X.T@d1); b1 -= lr*d1.sum(0,keepdims=True)
    return L
print("ritmo lr | perdida final (menor = mejor)")
for lr in [0.01, 0.1, 1.0, 5.0, 50.0]:
    print(f"  {lr:>5} | {perdida_final(lr):.4f}")
print("\nlr diminuto apenas aprende; lr enorme se desestabiliza. La zona buena esta en medio.")


## 12. Un problema realista: dos lunas entrelazadas

El XOR son 4 puntos. Subamos a un problema con cientos de ejemplos y ruido: dos «lunas» que se entrelazan
y **no** se pueden separar con una recta. Entrenamos una red $2\to8\to1$ y dibujamos su frontera curva.
Esto se parece mucho más a un caso real (clasificar dos clases que no están limpiamente separadas).


In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=0)
ym = ym.reshape(-1,1).astype(float)

rng = np.random.default_rng(1)
W1 = rng.standard_normal((2,8))*np.sqrt(2/2); b1 = np.zeros((1,8))
W2 = rng.standard_normal((8,1))*np.sqrt(2/8); b2 = np.zeros((1,1)); lr = 0.5
for _ in range(4000):
    a1 = np.tanh(Xm@W1+b1); p = sigmoide(a1@W2+b2)
    d2 = (p-ym)/len(Xm); d1 = (d2@W2.T)*(1-a1**2)
    W2 -= lr*(a1.T@d2); b2 -= lr*d2.sum(0,keepdims=True)
    W1 -= lr*(Xm.T@d1); b1 -= lr*d1.sum(0,keepdims=True)
acc = ((sigmoide(np.tanh(Xm@W1+b1)@W2+b2)>0.5).astype(float)==ym).mean()
print(f"Acierto de la red en las dos lunas: {acc:.3f}")

gx2,gy2 = np.meshgrid(np.linspace(Xm[:,0].min()-.5,Xm[:,0].max()+.5,300),
                      np.linspace(Xm[:,1].min()-.5,Xm[:,1].max()+.5,300))
m2 = np.c_[gx2.ravel(),gy2.ravel()]
pm2 = sigmoide(np.tanh(m2@W1+b1)@W2+b2).reshape(gx2.shape)
plt.figure(figsize=(6,4.2))
plt.contourf(gx2,gy2,pm2,levels=[0,.5,1],colors=["#eaeaea","#c4c4c4"])
plt.contour(gx2,gy2,pm2,levels=[.5],colors="black",linewidths=1.3)
plt.scatter(Xm[:,0],Xm[:,1],c=["black" if t else "white" for t in ym.ravel()],
            edgecolors="black",s=18)
plt.title(f"Dos lunas: la red encuentra una frontera curva (acierto {acc:.0%})")
plt.tight_layout(); plt.show()


**Esto ya es aprendizaje de verdad.** No le dijimos la forma de la frontera: la red la **dobló** para
seguir el contorno entre las dos lunas. La misma maquinaria (neurona, capa, retropropagación) que resolvió
4 puntos resuelve 300 con ruido. Cambia las lunas por píxeles o por *embeddings* de texto y tienes el
esqueleto de cualquier clasificador moderno.


## 13. Pruébalo tú

1. **Sube el ruido de las lunas** a `noise=0.4`: ¿sigue encontrando una frontera razonable? ¿Empieza a
   sobreajustar (frontera retorcida)? Conecta con el cuaderno del sobreajuste.
2. **Cambia las 8 neuronas** de las lunas a 2 (poca capacidad) y a 64 (mucha). ¿Cómo cambia la frontera?
3. **Usa ReLU** en las lunas en vez de tanh. ¿Aprende más rápido?
4. **Tres clases:** investiga `make_blobs` con 3 centros y una capa de salida con *softmax*. Es el paso
   natural hacia la clasificación multiclase.


## 14. Errores comunes

- **«No aprende, la pérdida no baja».** `lr` demasiado bajo o mala inicialización. Súbelo o cambia la
  semilla (lo viste en el experimento C).
- **«La pérdida explota / sale NaN».** `lr` demasiado alto: pasos enormes que desestabilizan. Bájalo.
- **«La sigmoide aprende lentísimo».** Se satura; usa tanh o ReLU en las capas ocultas (experimento B).
- **«Mi red de 2 neuronas a veces falla».** No es un bug: capacidad justa + azar (experimento A).
- **«Acierta el entrenamiento pero generaliza mal».** Sobreajuste: con las lunas y mucho ruido lo
  empiezas a ver. Ese es el tema del siguiente cuaderno de este facsímil.


## 15. Qué te llevas

- Una **neurona** traza una raya; para problemas que no son una raya —casi todos los interesantes—
  necesitas **capas**, y ahí nace la expresividad (XOR y lunas lo demuestran).
- **Entrenar** es medir el error con una **pérdida** y bajarlo a pasitos con un **optimizador**; la
  **retropropagación** (regla de la cadena) reparte el error capa por capa.
- Los **hiperparámetros** no son detalles: el número de neuronas, la **activación** (sigmoide vs tanh vs
  ReLU), el **ritmo** y el **azar** deciden si la red aprende o se atasca. Lo has medido tú.
- Lo escribiste **a mano**: PyTorch o TensorFlow hacen exactamente esto (forward, pérdida, gradientes,
  paso), solo que más rápido, con más capas y calculando las derivadas solas (*autograd*).

**Para seguir:** el cap. 8 (CNNs y RNNs) reutiliza estas ideas con otra forma; el cap. 9 (tokens y
*embeddings*) cambia los sensores por representaciones del lenguaje; y el facsímil 3 abre por dentro un
Transformer, que no es más que muchas de estas capas con la atención como pegamento.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*